In [1]:
! pip install ultralytics supervision roboflow inference

In [2]:
import os

os.getcwd()

'/content'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/Projects/Football_Analysis/

/content/drive/MyDrive/Projects/Football_Analysis


In [5]:
import numpy as np
import json
import os
import supervision as sv

from utils import read_video, save_video
from trackers import Tracker
from team_color_assigner import TeamColorAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from pitch_key_point import PitchKeyPoints , SoccerPitchConfiguration
from view_transformer import ViewTransformer
from pitch_key_point.draw_pitch import draw_pitch, draw_points_on_pitch
from utils import get_bbox_center, get_foot_position

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# Read the video file
video_frames = read_video('input_video/08fd33_41.mp4')

# Initialize tracker
tracker = Tracker(model_path = "models/player_detection/best11_v2.pt", track_threshold = 0.7)

# Get object tracks
tracks = tracker.get_object_tracks(video_frames,
                                   detect_conf = 0.1,
                                   read_from_stubs= False)

# Get object position
tracker.add_position_to_tracks(tracks)

# Camera movement estimator
camera_movement_estimator = CameraMovementEstimator(video_frames[0])
camera_movement_per_frame = camera_movement_estimator.get_camera_movement(video_frames,
                                                                          read_from_stub=False,
                                                                          stub_path='stubs/camera_movement_stub.pkl')
camera_movement_estimator.add_adjust_positions_to_tracks(tracks, camera_movement_per_frame)

# Interpolate ball positions
try:
  tracks['ball'] = tracker.interpolate_ball_positions(tracks['ball'])
except:
  pass

# Assign player teams
team_assigner = TeamColorAssigner()
team_assigner.assign_team_color(video_frames[3], tracks['players'][0])

for frame_num, player_track in enumerate(tracks['players']):
    for player_id, track in player_track.items():
        team = team_assigner.get_player_team(video_frames[frame_num],
                                              track['bbox'],
                                              player_id)
        tracks['players'][frame_num][player_id]['team'] = team
        tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors[team]


# Assign goalkeeper team
for frame_num, (goalkeepers_track, players_track) in enumerate(zip(tracks['goalkeepers'], tracks['players'])):
        team1_xy, team2_xy = [], []

        # Process each player's track in the current frame
        for player in players_track.values():
          centroid = get_foot_position(player['bbox'])
          if player['team'] == 1:
                team1_xy.append(centroid)
          elif player['team'] == 2:
                team2_xy.append(centroid)

        team1_xy_array = np.array(team1_xy)
        team2_xy_array = np.array(team2_xy)
        team1_centroid = np.mean(team1_xy, axis=0)
        team2_centroid = np.mean(team2_xy, axis=0)

        for goalkeeper in goalkeepers_track.values():
          try:
            centroid = get_foot_position(goalkeeper['bbox'])
            if np.linalg.norm(centroid - team1_centroid) < np.linalg.norm(centroid - team2_centroid):
              goalkeeper['team'] = 1
              goalkeeper['team_color'] = team_assigner.team_colors[1]
            else:
              goalkeeper['team'] = 2
              goalkeeper['team_color'] = team_assigner.team_colors[2]
          except:
            continue


# Assign referees color
get_referees_color = TeamColorAssigner()
referees_exist = True
for frame_num, referee_track in enumerate(tracks['referees']):
  if len(referee_track) > 0 and referees_exist == True:
    first_referee_id = list(tracks['referees'][frame_num].keys())[0]
    first_referee = tracks['referees'][frame_num][first_referee_id]['bbox']
    referees_color = np.array(get_referees_color.get_player_color(video_frames[0], first_referee))
    referees_exist = False


# Assign ball acquisition
player_assigner = PlayerBallAssigner()
team_ball_control = []

for frame_num, player_track in enumerate(tracks['players']):
    ball_bbox = tracks['ball'][frame_num][1]['bbox']
    assigned_player = player_assigner.assign_ball_to_player(player_track, ball_bbox)

    if assigned_player != -1:
        tracks['players'][frame_num][assigned_player]['has_ball'] = True
        team_ball_control.append(tracks['players'][frame_num][assigned_player]['team'])
    else:
      try:
        team_ball_control.append(team_ball_control[-1])
      except:
        continue

team_ball_control = np.array(team_ball_control)

# Draw output
## Draw object tracks
output_video_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control, referees_color)

# Draw camera movement
output_video_frames = camera_movement_estimator.draw_camera_movement(output_video_frames, camera_movement_per_frame)

SupervisionWarnings: `track_buffer` in `ByteTrack.__init__` is deprecated and will be remove in `supervision-0.23.0`. Use 'lost_track_buffer' instead.
SupervisionWarnings: `track_thresh` in `ByteTrack.__init__` is deprecated and will be remove in `supervision-0.23.0`. Use 'track_activation_threshold' instead.



0: 736x1280 20 players, 4 referees, 54.0ms
1: 736x1280 20 players, 4 referees, 54.0ms
2: 736x1280 1 ball, 20 players, 4 referees, 54.0ms
3: 736x1280 1 ball, 20 players, 4 referees, 54.0ms
4: 736x1280 1 ball, 20 players, 4 referees, 54.0ms
5: 736x1280 1 ball, 20 players, 4 referees, 54.0ms
6: 736x1280 1 ball, 21 players, 5 referees, 54.0ms
7: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
8: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
9: 736x1280 1 ball, 21 players, 3 referees, 54.0ms
10: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
11: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
12: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
13: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
14: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
15: 736x1280 2 balls, 20 players, 3 referees, 54.0ms
16: 736x1280 1 ball, 21 players, 3 referees, 54.0ms
17: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
18: 736x1280 1 ball, 20 players, 3 referees, 54.0ms
19: 736x1280 1 ball, 20 players, 3 r

In [7]:
# Pitch keypoint detection
path = os.getcwd()
api_key_path = os.path.join(path, 'training/Roboflow.json')
roboflow_api = json.load(open(api_key_path))
api_key = roboflow_api["api_key"]

pitch_key_point = PitchKeyPoints()
frame_points, pitch_points = pitch_key_point.key_point_detection(video_frames, api_key)

In [8]:
CONFIG = SoccerPitchConfiguration()
output_radar_frames = []

r, g, b = referees_color
referee_color = sv.Color(r,g,b)

for frame_num, frame in enumerate(output_video_frames):

  radar = draw_pitch(config=CONFIG)
  try:
    transformer = ViewTransformer(
      source=frame_points[frame_num].xy[0].astype(np.float32),
      target=pitch_points[frame_num].astype(np.float32)
      )

  except:
    continue

  radar_ball_bbox = tracks['ball'][frame_num][1]['bbox']
  radar_ball_center = np.array(get_bbox_center(radar_ball_bbox)).reshape(1, -1)
  radar_ball_center_transformed = transformer.transform_points(radar_ball_center)

  radar = draw_points_on_pitch(
        config=CONFIG,
        xy=radar_ball_center_transformed,
        face_color=sv.Color.WHITE,
        edge_color=sv.Color.BLACK,
        radius=10,
        pitch=radar)


  for player_id, player_track in tracks['players'][frame_num].items():
    player_track
    radar_player_bbox = player_track['bbox']
    radar_player_bbox_center = np.array(get_foot_position(radar_player_bbox)).reshape(1, -1)
    radar_player_bbox_center_transfromed = transformer.transform_points(radar_player_bbox_center)
    r, g, b = player_track['team_color']
    team_color = sv.Color(r,g,b)

    radar = draw_points_on_pitch(
        config=CONFIG,
        xy=radar_player_bbox_center_transfromed,
        face_color=team_color,
        edge_color=team_color,
        radius=15,
        pitch=radar)

  for referee_id, referee_track in tracks['referees'][frame_num].items():
    radar_referee_bbox = referee_track['bbox']
    radar_referee_bbox_center = np.array(get_foot_position(radar_referee_bbox)).reshape(1, -1)
    radar_referee_bbox_center_transfromed = transformer.transform_points(radar_referee_bbox_center)

    radar = draw_points_on_pitch(
        config=CONFIG,
        xy=radar_referee_bbox_center_transfromed,
        face_color=referee_color,
        edge_color=referee_color,
        radius=15,
        pitch=radar)

  h, w, _ = frame.shape
  annotated_frame = frame.copy()
  radar = sv.resize_image(radar, (w // 2, h // 2))
  radar_h, radar_w, _ = radar.shape
  rect = sv.Rect(
                x=w // 2 - radar_w // 2,
                y=h - radar_h,
                width=radar_w,
                height=radar_h
                )
  annotated_frame = sv.draw_image(annotated_frame, radar, opacity=0.5, rect=rect)

  output_radar_frames.append(annotated_frame)

In [9]:
# Save the video file
save_video(output_radar_frames, 'output_videos/output_radar_video.avi')

output_radar_video_3.avi released.
